# PyG Benchmark

Run this notebook in the **PyG environment**. It saves results to `comparison_outputs/` so `compare_results.ipynb` can compare them side-by-side with the DGL run.

Stages:
1. Graph construction — node/edge counts per type
2. Forward-pass parity — load DGL weights, compare single-pass scores *(requires `dgl_benchmark.ipynb` to have run first)*
3. Training — 1 pretrain epoch + 5 finetune epochs, save validation metrics
4. Disease-centric evaluation — per-disease AUROC on a sample of test diseases

In [ ]:
import sys, os, json, pickle
import torch
import numpy as np

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd()))
PYG_ROOT    = os.path.join(REPO_ROOT, 'pyg_implementation')
OUT_DIR     = os.path.join(REPO_ROOT, 'comparison_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, PYG_ROOT)
import txgnn

print('txgnn from:', txgnn.__file__)
print('Output dir:', OUT_DIR)

In [ ]:
# ── Shared config (must match dgl_benchmark.ipynb exactly) ──
DATA_FOLDER   = os.path.join(REPO_ROOT, 'data')
SPLIT         = 'complex_disease'
SEED          = 42
DEVICE        = 'cuda:0' if torch.cuda.is_available() else 'cpu'
N_HID         = 100
N_INP         = 100
N_OUT         = 100
PROTO         = True
PROTO_NUM     = 5
ATTENTION     = False
SIM_MEASURE   = 'all_nodes_profile'
AGG_MEASURE   = 'rarity'
N_PRETRAIN    = 1
N_FINETUNE    = 5
BATCH_SIZE    = 1024
LR            = 1e-3
SAMPLE_DISEASES = 5

DD_ETYPES = [
    ('drug', 'contraindication', 'disease'),
    ('drug', 'indication', 'disease'),
    ('drug', 'off-label use', 'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication', 'drug'),
    ('disease', 'rev_off-label use', 'drug'),
]

print('Device:', DEVICE)

---
## Stage 1 — Graph construction

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

data = txgnn.TxData(data_folder_path=DATA_FOLDER)
data.prepare_split(split=SPLIT, seed=SEED)
G = data.G

print('Graph loaded')
print('  node_types:', G.node_types)
print('  # edge types:', len(G.edge_types))

In [ ]:
graph_stats = {
    'node_counts': {ntype: G[ntype].num_nodes for ntype in G.node_types},
    'edge_counts': {str(et): G[et].num_edges for et in G.edge_types},
    'edge_indices': {},
}

for et in G.edge_types:
    ei = G[et].edge_index
    src = ei[0].numpy()
    dst = ei[1].numpy()
    idx = np.lexsort((dst, src))
    graph_stats['edge_indices'][str(et)] = {
        'src': src[idx].tolist(),
        'dst': dst[idx].tolist(),
    }

with open(os.path.join(OUT_DIR, 'pyg_graph_stats.json'), 'w') as f:
    json.dump(graph_stats, f)

print('Stage 1 saved: pyg_graph_stats.json')
print('  node types:', list(graph_stats['node_counts'].keys()))
print('  edge type count:', len(graph_stats['edge_counts']))

---
## Stage 2 — Forward-pass parity

Load the DGL model weights saved by `dgl_benchmark.ipynb`, copy them into the PyG model (parameter names are identical), run a single forward pass, and save the scores.

**Skip this stage if `dgl_benchmark.ipynb` hasn't been run yet** — move on to Stage 3.

In [ ]:
dgl_state_path = os.path.join(OUT_DIR, 'dgl_model_state.pt')
dgl_emb_path   = os.path.join(OUT_DIR, 'dgl_node_emb.pkl')
dgl_available  = os.path.exists(dgl_state_path) and os.path.exists(dgl_emb_path)
print('DGL weights available:', dgl_available)
if not dgl_available:
    print('Skipping forward-pass parity — run dgl_benchmark.ipynb first.')

In [ ]:
if dgl_available:
    torch.manual_seed(0)
    np.random.seed(0)

    model_fp = txgnn.TxGNN(data=data, device=DEVICE)
    model_fp.model_initialize(
        n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
        proto=PROTO, proto_num=PROTO_NUM,
        attention=ATTENTION,
        sim_measure=SIM_MEASURE,
        agg_measure=AGG_MEASURE,
    )

    # Load DGL weights into PyG model
    dgl_state = torch.load(dgl_state_path, map_location='cpu')
    pyg_state = model_fp.model.state_dict()

    common   = set(dgl_state.keys()) & set(pyg_state.keys())
    only_dgl = set(dgl_state.keys()) - set(pyg_state.keys())
    only_pyg = set(pyg_state.keys()) - set(dgl_state.keys())

    print(f'Common params: {len(common)}')
    if only_dgl: print(f'Only in DGL:   {sorted(only_dgl)}')
    if only_pyg: print(f'Only in PyG:   {sorted(only_pyg)}')

    shape_errors = []
    new_state = dict(pyg_state)
    with torch.no_grad():
        for name in common:
            if dgl_state[name].shape == pyg_state[name].shape:
                new_state[name] = dgl_state[name].clone()
            else:
                shape_errors.append((name, dgl_state[name].shape, pyg_state[name].shape))

    if shape_errors:
        print('Shape mismatches (skipped):')
        for n, sd, sp in shape_errors:
            print(f'  {n}: DGL={sd}  PyG={sp}')
    else:
        print('All weights copied successfully')

    model_fp.model.load_state_dict(new_state, strict=False)

    # Sync node embeddings
    with open(dgl_emb_path, 'rb') as f:
        dgl_emb = pickle.load(f)
    with torch.no_grad():
        for ntype in G.node_types:
            if ntype in dgl_emb:
                data.G[ntype].inp = dgl_emb[ntype].clone()
                model_fp.G[ntype].inp = dgl_emb[ntype].clone()
    print('Node embeddings synced from DGL')

In [ ]:
if dgl_available:
    from txgnn.utils import Full_Graph_NegSampler

    torch.manual_seed(1)
    G_dev = data.G.to(DEVICE)
    neg_sampler = Full_Graph_NegSampler(G_dev, 1, 'fix_dst', DEVICE)
    neg_G = neg_sampler(G_dev)

    model_fp.model.eval()
    with torch.no_grad():
        scores_pos, scores_neg, pos_out, neg_out = model_fp.model(
            G_dev, neg_G, pretrain_mode=False, mode='test'
        )

    forward_scores = {
        str(et): scores_pos[et].detach().cpu().tolist()
        for et in DD_ETYPES
        if et in scores_pos
    }

    with open(os.path.join(OUT_DIR, 'pyg_forward_scores.json'), 'w') as f:
        json.dump(forward_scores, f)

    print('Stage 2 saved: pyg_forward_scores.json')
    for et, scores in forward_scores.items():
        print(f'  {et}: {len(scores)} edges, mean score={float(np.mean(scores)):.4f}')

---
## Stage 3 — Training metrics

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

model = txgnn.TxGNN(data=data, device=DEVICE)
model.model_initialize(
    n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
    proto=PROTO, proto_num=PROTO_NUM,
    attention=ATTENTION,
    sim_measure=SIM_MEASURE,
    agg_measure=AGG_MEASURE,
)
print('Model initialized, params:', sum(p.numel() for p in model.model.parameters()))

In [ ]:
torch.manual_seed(0)
model.pretrain(
    n_epoch=N_PRETRAIN,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    train_print_per_n=9999,
)
print('Pretrain done')

In [ ]:
torch.manual_seed(0)
model.finetune(
    n_epoch=N_FINETUNE,
    learning_rate=LR,
    train_print_per_n=9999,
    valid_per_n=N_FINETUNE,
)
print('Finetune done')

In [ ]:
from txgnn.utils import evaluate_fb

(auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), loss = \
    evaluate_fb(model.best_model, model.g_valid_pos, model.g_valid_neg,
                data.G.to(DEVICE), DD_ETYPES, DEVICE)

metrics = {
    'macro_auroc': macro_auroc,
    'macro_auprc': macro_auprc,
    'micro_auroc': micro_auroc,
    'micro_auprc': micro_auprc,
    'loss': loss,
    'auroc_per_etype': {str(k): v for k, v in auroc_rel.items()},
    'auprc_per_etype': {str(k): v for k, v in auprc_rel.items()},
}

with open(os.path.join(OUT_DIR, 'pyg_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print('Stage 3 saved: pyg_metrics.json')
print(f'  Macro AUROC={macro_auroc:.4f}  Macro AUPRC={macro_auprc:.4f}  Loss={loss:.4f}')

---
## Stage 4 — Disease-centric evaluation

In [ ]:
evaluator = txgnn.TxEval(model=model, data=data)
disease_ids = evaluator.retrieve_disease_idxs_test_set('indication')[:SAMPLE_DISEASES]
print('Evaluating diseases:', disease_ids.tolist())

result = evaluator.eval_disease_centric(
    disease_idxs=disease_ids.tolist(),
    relation='indication',
    return_raw=True,
    show_plot=False,
    verbose=False,
    simulate_random=False,
)

disease_auroc = {str(k): float(v) for k, v in result['result']['AUROC'].items()}

with open(os.path.join(OUT_DIR, 'pyg_disease_auroc.json'), 'w') as f:
    json.dump({'disease_ids': disease_ids.tolist(), 'auroc': disease_auroc}, f, indent=2)

print('Stage 4 saved: pyg_disease_auroc.json')
for did, auc in disease_auroc.items():
    print(f'  disease {did}: AUROC={auc:.4f}')

---
All outputs saved to `comparison_outputs/`. Open `compare_results.ipynb` to see the side-by-side comparison.